<a href="https://colab.research.google.com/github/Sandutta2020/Pyspark/blob/Master/pyspark_diabetes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
!pip install pyspark

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("spark").getOrCreate()

In [0]:
!git clone https://github.com/education454/diabetes_dataset

In [0]:
!ls diabetes_dataset


In [0]:
df =spark.read.csv("/content/diabetes_dataset/diabetes.csv", header=True, inferSchema=True)

In [0]:
df.show()

In [0]:
df.printSchema()

In [0]:
print((df.count(), len(df.columns)))

In [0]:
df.describe().show()

In [0]:
df.groupby("Outcome").count().show()

In [0]:
for col in df.columns:
  print(col+":", df[df[col].isNull()].count())

In [0]:
def count_zeros():
  columns_list = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
  for i in columns_list:
    print(i+":", df[df[i]==0].count())


In [0]:
count_zeros()

In [0]:
for i in df.columns[1:6]:
    data =df.agg({i:"mean"}).first()[0]
    print("mean value for {} is {}".format(i, int(data)))

In [0]:
from pyspark.sql.functions import *
for i in df.columns[1:6]:
    data =df.agg({i:"mean"}).first()[0]
    print("mean value for {} is {}".format(i, int(data)))
    df = df.withColumn(i, when(col(i)==0, int(data)).otherwise(col(i)))

In [0]:
df.show()

In [0]:
for col in df.columns:
  print("Correlation to outcome for", col, df.stat.corr("Outcome", col))

In [0]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"], outputCol="features")

In [0]:
output_data=assembler.transform(df)

In [0]:
output_data.printSchema()

In [0]:
output_data.show()

In [0]:
from pyspark.ml.classification import LogisticRegression
final_data = output_data.select("features", "Outcome")

In [0]:
train, test = final_data.randomSplit([0.7, 0.3])
models = LogisticRegression(labelCol="Outcome")
model = models.fit(train)

In [0]:
summary =model.summary

In [0]:
summary.predictions.describe().show()

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
predictions = model.evaluate(test)

In [0]:
predictions.predictions.show(20)

In [0]:
evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="Outcome")
evaluator.evaluate(model.transform(test))

In [0]:
model.save("model")

In [0]:
!ls

In [0]:
import pyspark.ml.classification as classification
model = classification.LogisticRegressionModel.load("model")